In [9]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

print(train.shape, val.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 35) (133316, 35)


In [14]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\soohok402\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [12]:
# ============================================================
# 1. 급락 특이 관측소 라벨 생성
# ============================================================

rapid_drop_stations = ['ST0358', 'ST0292', 'ST0326']

train['rapid_drop_station'] = (
    train['station_id'].isin(rapid_drop_stations).astype(int)
)

val['rapid_drop_station'] = (
    val['station_id'].isin(rapid_drop_stations).astype(int)
)

# 라벨 분포 확인
print(train['rapid_drop_station'].value_counts())
print(val['rapid_drop_station'].value_counts())

rapid_drop_station
0    612181
1      4112
Name: count, dtype: int64
rapid_drop_station
0    132425
1       891
Name: count, dtype: int64


In [13]:
# ============================================================
# 2. 입력 피처 구성
# ============================================================

feature_cols = num_cols_fe + ['station_id', 'rapid_drop_station']

X_tr = train[feature_cols].copy()
X_va = val[feature_cols].copy()

y_tr = train['target_maxrise_6h']
y_va = val['target_maxrise_6h']

In [15]:
# ============================================================
# 3. CatBoost 모델 학습 및 홀드아웃 평가
# ============================================================

model_cat_rapid = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id'],
    verbose=200
    # 나머지 하이퍼파라미터는 기존 모델과 동일하게 설정
)

model_cat_rapid.fit(
    X_tr, y_tr,
    eval_set=(X_va, y_va),
    early_stopping_rounds=200
)

pred_tr = model_cat_rapid.predict(X_tr)
pred_va = model_cat_rapid.predict(X_va)

rmse_tr = root_mean_squared_error(y_tr, pred_tr)
rmse_va = root_mean_squared_error(y_va, pred_va)

print(f"Train RMSE: {rmse_tr:.6f}")
print(f"Val RMSE  : {rmse_va:.6f}")
print(f"Best iteration: {model_cat_rapid.get_best_iteration()}")

Learning rate set to 0.071425
0:	learn: 0.3895344	test: 0.3445603	best: 0.3445603 (0)	total: 220ms	remaining: 10m 58s
200:	learn: 0.2300853	test: 0.2030500	best: 0.2030500 (200)	total: 19s	remaining: 4m 25s
400:	learn: 0.2182405	test: 0.1972399	best: 0.1972399 (400)	total: 37.7s	remaining: 4m 4s
600:	learn: 0.2109572	test: 0.1951762	best: 0.1951762 (600)	total: 56.1s	remaining: 3m 43s
800:	learn: 0.2055694	test: 0.1939608	best: 0.1939608 (800)	total: 1m 14s	remaining: 3m 24s
1000:	learn: 0.2011383	test: 0.1937466	best: 0.1937299 (996)	total: 1m 33s	remaining: 3m 5s
1200:	learn: 0.1973682	test: 0.1934138	best: 0.1934138 (1200)	total: 1m 51s	remaining: 2m 46s
1400:	learn: 0.1941003	test: 0.1932998	best: 0.1932998 (1400)	total: 2m 9s	remaining: 2m 27s
1600:	learn: 0.1912433	test: 0.1931135	best: 0.1930701 (1568)	total: 2m 27s	remaining: 2m 9s
1800:	learn: 0.1885726	test: 0.1931571	best: 0.1930565 (1635)	total: 2m 46s	remaining: 1m 50s
Stopped by overfitting detector  (200 iterations wait)